# Inspecting environment specifications and initial state

This notebook inventories the configurable boundaries of the three supported environments, then shows how to verify the meanings of the fields recorded by `capture_initial_state()`. There are three related but distinct things to inspect:

1. `env.spec` describes how an environment is registered (entry point, time limit, and constructor arguments).
2. `env.observation_space` describes the observation presented to a policy.
3. `env.unwrapped` exposes the underlying simulator implementation and its internal state.

The third item matters here because `state`, `agent_pos`, and `agent_dir` are implementation attributes rather than fields guaranteed by Gymnasium's common `Env` interface. Always inspect them after calling `reset()`.

In [1]:
import inspect
from importlib import import_module
from importlib.metadata import version

import gymnasium as gym
import numpy as np

from active_eval_gym.envs.factory import (
    SUPPORTED_ENVIRONMENTS,
    capture_initial_state,
    make_environment,
)

print("gymnasium:", version("gymnasium"))
print("minigrid:", version("minigrid"))

gymnasium: 1.3.0
minigrid: 3.1.0


## 1. System parameters, actions, and episode boundaries

The values below are the nominal values in the installed Gymnasium 1.3.0 and MiniGrid 3.1.0 implementations. A parameter appearing as an attribute does **not** automatically make it a supported constructor argument.

| Environment | Nominal system parameters | Action boundary | Termination / truncation boundary |
|---|---|---|---|
| CartPole-v1 | gravity `9.8`; cart mass `1.0`; pole mass `0.1`; `length=0.5` is half the pole length, so the full pole is `1.0`; force magnitude `10`; update interval `tau=0.02 s` | `Discrete(2)`: `0` applies `-10`, `1` applies `+10` | terminates at cart position beyond `+/-2.4` or pole angle beyond `+/-12 degrees`; wrapper truncates at 500 steps |
| Pendulum-v1 | gravity `10`; mass `1`; length `1`; update interval `dt=0.05 s`; maximum speed `8` | one-dimensional torque in `[-2, 2]`; inputs are clipped to this range | no state-based termination; wrapper truncates at 200 steps |
| MiniGrid-Empty-8x8-v0 | `8 x 8` grid; `7 x 7` agent view; `max_steps=4 * size^2=256` | `Discrete(7)`: left, right, forward, pickup, drop, toggle, done | reaching the goal terminates; 256 logical actions truncates |

The CartPole observation limits (`+/-4.8` position and `+/-24 degrees` angle) are wider than its termination limits so that the final failing observation remains representable.

In [2]:
parameter_names = {
    "CartPole-v1": (
        "gravity",
        "masscart",
        "masspole",
        "total_mass",
        "length",
        "polemass_length",
        "force_mag",
        "tau",
        "kinematics_integrator",
        "theta_threshold_radians",
        "x_threshold",
    ),
    "Pendulum-v1": ("g", "m", "l", "dt", "max_speed", "max_torque"),
    "MiniGrid-Empty-8x8-v0": (
        "width",
        "height",
        "agent_view_size",
        "max_steps",
        "agent_start_pos",
        "agent_start_dir",
    ),
}

for env_id, names in parameter_names.items():
    env = make_environment(env_id)
    try:
        base_env = env.unwrapped
        print(f"\n{env_id}")
        print("  constructor:", inspect.signature(type(base_env).__init__))
        print("  reset:", inspect.signature(type(base_env).reset))
        print("  action space:", env.action_space)
        print("  render FPS:", base_env.metadata.get("render_fps"))
        print("  parameters:")
        for name in names:
            print(f"    {name}: {getattr(base_env, name)!r}")
        if hasattr(base_env, "actions"):
            print(
                "  actions:",
                [(action.value, action.name) for action in base_env.actions],
            )
    finally:
        env.close()


CartPole-v1
  constructor: (self, sutton_barto_reward: bool = False, render_mode: str | None = None)
  reset: (self, *, seed: int | None = None, options: dict | None = None)
  action space: Discrete(2)
  render FPS: 50
  parameters:
    gravity: 9.8
    masscart: 1.0
    masspole: 0.1
    total_mass: 1.1
    length: 0.5
    polemass_length: 0.05
    force_mag: 10.0
    tau: 0.02
    kinematics_integrator: 'euler'
    theta_threshold_radians: 0.20943951023931953
    x_threshold: 2.4

Pendulum-v1
  constructor: (self, render_mode: str | None = None, g=10.0)
  reset: (self, *, seed: int | None = None, options: dict | None = None)
  action space: Box(-2.0, 2.0, (1,), float32)
  render FPS: 30
  parameters:
    g: 10.0
    m: 1.0
    l: 1.0
    dt: 0.05
    max_speed: 8
    max_torque: 2.0

MiniGrid-Empty-8x8-v0
  constructor: (self, size=8, agent_start_pos=(1, 1), agent_start_dir=0, max_steps: 'int | None' = None, **kwargs)
  reset: (self, *, seed: 'int | None' = None, options: 'dict[str,

## 2. Action rate versus rendering and evaluation rate

CartPole advances simulated time by `tau=0.02 s` per action, giving a nominal control rate of 50 Hz. Pendulum advances by `dt=0.05 s`, giving 20 Hz. Consequently, both have a maximum of 10 simulated seconds per episode under their standard time limits.

MiniGrid has no physical-time interval: one action advances one logical grid-world step. Its `render_fps=10` controls only human rendering speed. More generally, `render_fps` is not a control frequency—Pendulum renders at 30 FPS but its dynamics update at 20 Hz.

The repository's `collect_episode()` requests one policy action for every `env.step()` call. It has no action-repeat or independent evaluation-frequency option. The frequency at which an outer active evaluator chooses a new environment is an experiment-level decision, not an environment parameter, and is not implemented yet.

In [3]:
timing_rows = []
for env_id, interval_name in (("CartPole-v1", "tau"), ("Pendulum-v1", "dt")):
    env = make_environment(env_id)
    try:
        interval = getattr(env.unwrapped, interval_name)
        max_steps = env.spec.max_episode_steps
        timing_rows.append(
            (env_id, interval, 1 / interval, max_steps, interval * max_steps)
        )
    finally:
        env.close()

for env_id, interval, control_hz, max_steps, max_seconds in timing_rows:
    print(
        f"{env_id}: one action every {interval:g} simulated seconds "
        f"({control_hz:g} Hz), {max_steps} steps = {max_seconds:g} simulated seconds"
    )

env = make_environment("MiniGrid-Empty-8x8-v0")
try:
    print(
        f"MiniGrid-Empty-8x8-v0: no physical-time rate; "
        f"{env.unwrapped.max_steps} logical action steps; "
        f"render_fps={env.unwrapped.metadata['render_fps']} is visualization only"
    )
finally:
    env.close()

CartPole-v1: one action every 0.02 simulated seconds (50 Hz), 500 steps = 10 simulated seconds
Pendulum-v1: one action every 0.05 simulated seconds (20 Hz), 200 steps = 10 simulated seconds
MiniGrid-Empty-8x8-v0: no physical-time rate; 256 logical action steps; render_fps=10 is visualization only


## 3. Configurable initial-state boundaries

### CartPole

`reset(options={"low": ..., "high": ...})` samples **all four** internal state entries independently from the same interval. The default is `[-0.05, 0.05]`. Gymnasium validates only that `low <= high`; it warns that custom ranges can produce an already-failing or out-of-observation-space initial state. There is no public reset option for four separate per-state intervals.

### Pendulum

`reset(options={"x_init": ..., "y_init": ...})` uses symmetric ranges: angle in `[-x_init, x_init]` and angular velocity in `[-y_init, y_init]`. Defaults are `x_init=pi` and `y_init=1`. These values are range half-widths, not exact initial values.

### MiniGrid Empty

The start is configured at construction, not through reset options. `agent_start_pos=(x, y)` fixes the position; `None` requests random placement. `agent_start_dir` uses `0=right`, `1=down`, `2=left`, `3=up`. On an 8x8 Empty grid, ordinary free interior coordinates have `x` and `y` in `1..6`; `(6, 6)` contains the goal and should not be used as a nominal start. The default is `(1, 1)`, facing right.

In [4]:
# CartPole: every state component is sampled from the same custom interval.
cartpole = gym.make("CartPole-v1")
try:
    cartpole.reset(seed=123, options={"low": -0.01, "high": 0.01})
    cartpole_state = capture_initial_state(cartpole, "CartPole-v1")
    print("CartPole custom reset:", cartpole_state)
    assert all(-0.01 <= value <= 0.01 for value in cartpole_state.values())
finally:
    cartpole.close()

# Pendulum: angle and angular-velocity ranges have separate symmetric half-widths.
pendulum = gym.make("Pendulum-v1")
try:
    pendulum.reset(seed=123, options={"x_init": 0.25, "y_init": 0.1})
    pendulum_state = capture_initial_state(pendulum, "Pendulum-v1")
    print("Pendulum custom reset:", pendulum_state)
    assert abs(pendulum_state["angle"]) <= 0.25
    assert abs(pendulum_state["angular_velocity"]) <= 0.1
finally:
    pendulum.close()

# MiniGrid: start controls are constructor arguments and remain fixed across resets.
minigrid_env = gym.make(
    "MiniGrid-Empty-8x8-v0",
    agent_start_pos=(2, 3),
    agent_start_dir=3,
)
try:
    minigrid_env.reset(seed=123)
    minigrid_state = capture_initial_state(minigrid_env, "MiniGrid-Empty-8x8-v0")
    print("MiniGrid custom fixed start:", minigrid_state)
    assert minigrid_state == {"agent_position": [2, 3], "agent_direction": 3}
finally:
    minigrid_env.close()

CartPole custom reset: {'cart_position': 0.00364703726496287, 'cart_velocity': -0.008923579623955546, 'pole_angle': -0.005592802544547772, 'pole_angular_velocity': -0.006312563786026606}
Pendulum custom reset: {'angle': 0.09117593162407173, 'angular_velocity': -0.08923579623955546}
MiniGrid custom fixed start: {'agent_position': [2, 3], 'agent_direction': 3}


## 4. What the repository currently exposes

The examples above demonstrate capabilities of the upstream environments. They are not yet experiment configurations in this repository:

- `make_environment()` accepts an environment ID and a `PerturbationSpec`, but no constructor arguments.
- `collect_episode()` calls `reset(seed=episode_seed)` without reset options.
- The only implemented perturbation is currently `none`.

Therefore, changing a physical parameter or initial-state distribution for a real evaluation should be added as an explicit, serializable perturbation spec and applied through a reusable wrapper. Avoid scattered assignments to `env.unwrapped`: several values are coupled. For example, changing CartPole's pole mass or half-length also requires recomputing `total_mass` and/or `polemass_length`. This keeps nominal reproduction, metadata, and comparisons reliable.

## 5. Inspect the registered specifications

A registered spec identifies the concrete implementation and wrapper-level settings. MiniGrid must be imported before its environments appear in Gymnasium's registry.

In [5]:
import_module("minigrid")

for env_id in SUPPORTED_ENVIRONMENTS:
    spec = gym.spec(env_id)
    print(f"{env_id}:")
    print(f"  entry point: {spec.entry_point}")
    print(f"  maximum episode steps: {spec.max_episode_steps}")
    print(f"  constructor arguments: {spec.kwargs}")

CartPole-v1:
  entry point: gymnasium.envs.classic_control.cartpole:CartPoleEnv
  maximum episode steps: 500
  constructor arguments: {}
Pendulum-v1:
  entry point: gymnasium.envs.classic_control.pendulum:PendulumEnv
  maximum episode steps: 200
  constructor arguments: {}
MiniGrid-Empty-8x8-v0:
  entry point: minigrid.envs:EmptyEnv
  maximum episode steps: None
  constructor arguments: {}


## 6. Compare observations with internal state

The observation is what the policy receives; it is not necessarily the complete or direct simulator state. In particular, Pendulum exposes `[cos(angle), sin(angle), angular_velocity]` while its internal state is `[angle, angular_velocity]`. MiniGrid exposes a partially observed image plus direction and mission, while the agent's world position remains internal.

In [6]:
def summarize_observation(observation):
    if isinstance(observation, dict):
        return {
            key: (
                f"array(shape={value.shape}, dtype={value.dtype})"
                if isinstance(value, np.ndarray)
                else value
            )
            for key, value in observation.items()
        }
    return observation


for env_id in SUPPORTED_ENVIRONMENTS:
    env = make_environment(env_id)
    try:
        observation, reset_info = env.reset(seed=123)
        base_env = env.unwrapped

        print(f"\n{env_id}")
        print(
            "  implementation:",
            f"{type(base_env).__module__}.{type(base_env).__qualname__}",
        )
        print("  source file:", inspect.getsourcefile(type(base_env)))
        print("  observation space:", env.observation_space)
        print("  reset observation:", summarize_observation(observation))
        print("  raw state attribute:", getattr(base_env, "state", "not present"))
        print("  agent position:", getattr(base_env, "agent_pos", "not present"))
        print("  agent direction:", getattr(base_env, "agent_dir", "not present"))
        print("  captured state:", capture_initial_state(env, env_id))
    finally:
        env.close()


CartPole-v1
  implementation: gymnasium.envs.classic_control.cartpole.CartPoleEnv
  source file: /home/chesi/active-eval-gym/.venv/lib/python3.12/site-packages/gymnasium/envs/classic_control/cartpole.py
  observation space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
  reset observation: [ 0.01823519 -0.0446179  -0.02796401 -0.03156282]
  raw state attribute: [ 0.01823519 -0.0446179  -0.02796401 -0.03156282]
  agent position: not present
  agent direction: not present
  captured state: {'cart_position': 0.018235186324814343, 'cart_velocity': -0.04461789811977773, 'pole_angle': -0.027964012722738865, 'pole_angular_velocity': -0.03156281893013303}

Pendulum-v1
  implementation: gymnasium.envs.classic_control.pendulum.PendulumEnv
  source file: /home/chesi/active-eval-gym/.venv/lib/python3.12/site-packages/gymnasium/envs/classic_control/pendulum.py
  observation space: Box([-1. -1. -8.], [1. 1. 8.], (3,), float32)
 

## 7. Inspect the installed source

Package documentation describes the public observation, but the installed source is the strongest evidence for private attribute order in the exact versions used by this project. The following helper prints relevant lines and their installed-source line numbers.

In [7]:
from minigrid.core.constants import DIR_TO_VEC


def print_matching_source(cls, patterns):
    lines, first_line = inspect.getsourcelines(cls)
    source_file = inspect.getsourcefile(cls)
    print(f"\n{cls.__qualname__}: {source_file}")
    for offset, line in enumerate(lines):
        if any(pattern in line for pattern in patterns):
            print(f"  {first_line + offset}: {line.rstrip()}")

source_patterns = {
    "CartPole-v1": ("x, x_dot, theta, theta_dot = self.state", "self.state ="),
    "Pendulum-v1": (
        "th, thdot = self.state",
        "theta, thetadot = self.state",
        "self.state =",
    ),
    "MiniGrid-Empty-8x8-v0": ("self.agent_pos =", "self.agent_dir ="),
}

for env_id, patterns in source_patterns.items():
    env = make_environment(env_id)
    try:
        print_matching_source(type(env.unwrapped), patterns)
    finally:
        env.close()

direction_names = ("right", "down", "left", "up")
print("\nMiniGrid direction encoding:")
for index, (name, vector) in enumerate(zip(direction_names, DIR_TO_VEC, strict=True)):
    print(f"  {index}: {name:>5} -> {vector.tolist()}")


CartPoleEnv: /home/chesi/active-eval-gym/.venv/lib/python3.12/site-packages/gymnasium/envs/classic_control/cartpole.py
  168:         x, x_dot, theta, theta_dot = self.state
  195:         self.state = np.array((x, x_dot, theta, theta_dot), dtype=np.float64)
  241:         self.state = self.np_random.uniform(low=low, high=high, size=(4,))

PendulumEnv: /home/chesi/active-eval-gym/.venv/lib/python3.12/site-packages/gymnasium/envs/classic_control/pendulum.py
  127:         th, thdot = self.state  # th := theta
  142:         self.state = np.array([newth, newthdot])
  162:         self.state = self.np_random.uniform(low=low, high=high)
  170:         theta, thetadot = self.state

EmptyEnv: /home/chesi/active-eval-gym/.venv/lib/python3.12/site-packages/minigrid/envs/empty.py
  109:             self.agent_pos = self.agent_start_pos
  110:             self.agent_dir = self.agent_start_dir

MiniGrid direction encoding:
  0: right -> [1, 0]
  1:  down -> [0, 1]
  2:  left -> [-1, 0]
  3:    u

## 8. Executable semantic checks

These assertions protect the assumptions made by `capture_initial_state()`. They are more useful than checking only the number of state values because they verify the relationship between the recorded state and the public observation.

In [8]:
# CartPole's observation directly represents its four internal state values.
env = make_environment("CartPole-v1")
try:
    observation, _ = env.reset(seed=123)
    captured = capture_initial_state(env, "CartPole-v1")
    recorded = np.array(
        [
            captured["cart_position"],
            captured["cart_velocity"],
            captured["pole_angle"],
            captured["pole_angular_velocity"],
        ]
    )
    np.testing.assert_allclose(observation, recorded, rtol=1e-6, atol=1e-7)
finally:
    env.close()

# Pendulum transforms the internal angle into cosine and sine.
env = make_environment("Pendulum-v1")
try:
    observation, _ = env.reset(seed=123)
    captured = capture_initial_state(env, "Pendulum-v1")
    expected_observation = np.array(
        [
            np.cos(captured["angle"]),
            np.sin(captured["angle"]),
            captured["angular_velocity"],
        ]
    )
    np.testing.assert_allclose(observation, expected_observation, rtol=1e-6, atol=1e-7)
finally:
    env.close()

# Empty-8x8 has a fixed nominal start at world coordinate (1, 1), facing right.
env = make_environment("MiniGrid-Empty-8x8-v0")
try:
    observation, _ = env.reset(seed=123)
    captured = capture_initial_state(env, "MiniGrid-Empty-8x8-v0")
    assert captured["agent_position"] == [1, 1]
    assert captured["agent_direction"] == 0
    assert observation["direction"] == captured["agent_direction"]
finally:
    env.close()

print("All initial-state semantic checks passed.")

All initial-state semantic checks passed.


## Maintenance note

Run these checks after dependency upgrades. `env.spec` and `observation_space` are public Gymnasium concepts, but this repository deliberately records additional interpretable simulator state through implementation attributes. Keeping package versions in episode metadata and testing these semantic relationships makes that dependency explicit and traceable.